In [0]:
# ---- Config ----
CATALOG = "workspace"
SILVER_TABLE = f"{CATALOG}.careerpulse_silver.job_postings"
EMBEDDINGS_TABLE = f"{CATALOG}.careerpulse_gold.job_description_embeddings"

In [0]:
import re
import pandas as pd
from bs4 import BeautifulSoup

def clean_description(html: str) -> str:
    if not html:
        return ""
    
    # 1. Strip HTML tags
    text = BeautifulSoup(html, "html.parser").get_text(separator=" ")
    
    # 2. Lowercase
    text = text.lower()
    
    # 3. Remove URLs
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    
    # 4. Remove email addresses
    text = re.sub(r"\S+@\S+", " ", text)
    
    # 5. Remove special characters and digits, keep only letters and spaces
    text = re.sub(r"[^a-z\s]", " ", text)
    
    # 6. Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [0]:
# Load silver, pull only the columns we need
silver_pd = (
    spark.table(SILVER_TABLE)
    .select("posting_id", "category", "title", "description")
    .toPandas()
)

silver_pd = silver_pd[silver_pd["category"]!="Unknown"]

print(f"Total rows:        {len(silver_pd)}")
print(f"Labeled (train):   {silver_pd['category'].notna().sum()}")
print(f"Unlabeled (infer): {silver_pd['category'].isna().sum()}")

In [0]:
# Apply cleaning
silver_pd["text_input"] = silver_pd["description"].apply(clean_description)

# Sanity checks
empty_after_clean = (silver_pd["text_input"].str.strip() == "").sum()
avg_token_count   = silver_pd["text_input"].str.split().apply(len).mean()

print(f"Empty after cleaning:  {empty_after_clean}")
print(f"Avg token count:       {avg_token_count:.0f}")

# Spot check
silver_pd[["title", "category", "text_input"]].sample(3)

In [0]:
# Drop rows where cleaning produced empty text — can't embed these
silver_pd = silver_pd[silver_pd["text_input"].str.strip() != ""]

# Split labeled vs unlabeled for later use
labeled_df   = silver_pd[silver_pd["category"].notna()].copy()
unlabeled_df = silver_pd[silver_pd["category"].isna()].copy()

print(f"Labeled rows available for training: {len(labeled_df)}")
print(f"Unlabeled rows to predict:           {len(unlabeled_df)}")

In [0]:
# ---- Shared imports ----
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
import pandas as pd

In [0]:
# ---- Shared eval function ----
# Uses stratified 5-fold CV to compare approaches on the labeled set

def evaluate_pipeline(pipeline, X, y, label=""):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y, cv=cv, scoring="accuracy")
    print(f"{label}")
    print(f"  Fold accuracies: {[round(s, 3) for s in scores]}")
    print(f"  Mean: {scores.mean():.3f}  |  Std: {scores.std():.3f}\n")
    return scores

In [0]:
X = labeled_df["text_input"].values
y = labeled_df["category"].values

In [0]:
# ---- Option 1: TF-IDF + KNN (baseline) ----

pipeline_1 = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=10_000,
        ngram_range=(1, 2),   # unigrams + bigrams
        sublinear_tf=True,    # dampens effect of very high term frequencies
        min_df=2              # ignore terms appearing in fewer than 2 docs
    )),
    ("knn", KNeighborsClassifier(
        n_neighbors=5,
        metric="cosine"       # cosine similarity is better than euclidean for text
    ))
])

scores_1 = evaluate_pipeline(pipeline_1, X, y, label="Option 1: TF-IDF + KNN")

In [0]:
# ---- Option 2: TF-IDF + SVD + KNN ----

pipeline_2 = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=10_000,
        ngram_range=(1, 2),
        sublinear_tf=True,
        min_df=2
    )),
    ("svd", TruncatedSVD(
        n_components=200,     # experiment with 100, 200, 300
        random_state=42
    )),
    ("knn", KNeighborsClassifier(
        n_neighbors=5,
        metric="cosine"
    ))
])

scores_2 = evaluate_pipeline(pipeline_2, X, y, label="Option 2: TF-IDF + SVD + KNN")

In [0]:
# %pip install sentence_transformers

In [0]:
# ---- Option 3: Sentence Transformer + KNN ----
# Downloads ~80MB model on first run

from sentence_transformers import SentenceTransformer

MODEL_NAME = "all-MiniLM-L6-v2"  # fast, good quality, 384-dim output
model = SentenceTransformer(MODEL_NAME)

# Generate embeddings for labeled set
# encode() returns a numpy array of shape (n_samples, 384)
X_st = model.encode(
    labeled_df["text_input"].tolist(),
    batch_size=64,
    show_progress_bar=True
)

knn_3 = KNeighborsClassifier(n_neighbors=5, metric="cosine")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_3 = cross_val_score(knn_3, X_st, y, cv=cv, scoring="accuracy")

print("Option 3: Sentence Transformer + KNN")
print(f"  Fold accuracies: {[round(s, 3) for s in scores_3]}")
print(f"  Mean: {scores_3.mean():.3f}  |  Std: {scores_3.std():.3f}\n")

In [0]:
# ---- Summary comparison ----

summary = pd.DataFrame({
    "approach": [
        "TF-IDF + KNN",
        "TF-IDF + SVD + KNN",
        "Sentence Transformer + KNN"
    ],
    "mean_accuracy": [scores_1.mean(), scores_2.mean(), scores_3.mean()],
    "std":           [scores_1.std(),  scores_2.std(),  scores_3.std()]
}).sort_values("mean_accuracy", ascending=False)

display(summary)

In [0]:
# 1. Concatenate title into text_input — titles are dense category signals
labeled_df["text_input"] = labeled_df["title"] + " " + labeled_df["text_input"]

# 2. Try different values of k
for k in [3, 5, 7, 11, 15]:
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=10_000, ngram_range=(1,2), sublinear_tf=True, min_df=2)),
        ("knn", KNeighborsClassifier(n_neighbors=k, metric="cosine"))
    ])
    scores = cross_val_score(pipeline, X, y, cv=cv, scoring="accuracy")
    print(f"k={k}:  mean={scores.mean():.3f}  std={scores.std():.3f}")

In [0]:
# 3. Try weighted voting — closer neighbors vote more strongly
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=10_000, ngram_range=(1,2), sublinear_tf=True, min_df=2)),
    ("knn", KNeighborsClassifier(n_neighbors=5, metric="cosine", weights="distance"))
])
scores = cross_val_score(pipeline, X, y, cv=cv, scoring="accuracy")
print(f"distance weighting:  mean={scores.mean():.3f}  std={scores.std():.3f}")

In [0]:
labeled_df["category"].value_counts(dropna=False)

In [0]:
# 4. Check per-category accuracy — overall mean can hide badly performing categories
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_predict

y_pred = cross_val_predict(pipeline_1, X, y, cv=cv)
print(classification_report(y, y_pred))

In [0]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import classification_report

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ---- 1. Sweep k ----
print("=== k sweep ===")
for k in [3, 5, 7, 11, 15]:
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=10_000, ngram_range=(1,2), sublinear_tf=True, min_df=2)),
        ("knn", KNeighborsClassifier(n_neighbors=k, metric="cosine"))
    ])
    scores = cross_val_score(pipeline, X, y, cv=cv, scoring="accuracy")
    print(f"  k={k}:  mean={scores.mean():.3f}  std={scores.std():.3f}")

# ---- 2. Distance weighting ----
print("\n=== distance weighting (k=5) ===")
pipeline_dist = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=10_000, ngram_range=(1,2), sublinear_tf=True, min_df=2)),
    ("knn", KNeighborsClassifier(n_neighbors=5, metric="cosine", weights="distance"))
])
scores_dist = cross_val_score(pipeline_dist, X, y, cv=cv, scoring="accuracy")
print(f"  mean={scores_dist.mean():.3f}  std={scores_dist.std():.3f}")

# ---- 3. Title prepended twice ----
print("\n=== title boosting ===")
X_boosted = (labeled_df["title"] + " " + labeled_df["title"] + " " + labeled_df["text_input"]).values
pipeline_boost = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=10_000, ngram_range=(1,2), sublinear_tf=True, min_df=2)),
    ("knn", KNeighborsClassifier(n_neighbors=5, metric="cosine"))
])
scores_boost = cross_val_score(pipeline_boost, X_boosted, y, cv=cv, scoring="accuracy")
print(f"  mean={scores_boost.mean():.3f}  std={scores_boost.std():.3f}")

# ---- 4. Combined: title boosting + distance weighting + best k ----
print("\n=== combined: title boost + distance weighting (k=5) ===")
pipeline_combined = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=10_000, ngram_range=(1,2), sublinear_tf=True, min_df=2)),
    ("knn", KNeighborsClassifier(n_neighbors=5, metric="cosine", weights="distance"))
])
scores_combined = cross_val_score(pipeline_combined, X_boosted, y, cv=cv, scoring="accuracy")
print(f"  mean={scores_combined.mean():.3f}  std={scores_combined.std():.3f}")

In [0]:
%pip install umap-learn plotly

In [0]:
# ---- 3. UMAP to 3D ----
reducer = UMAP(
    n_components=3,     # changed from 2
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=42
)
X_3d = reducer.fit_transform(X_svd)

# ---- 4. Plot 3D ----
plot_df = pd.DataFrame({
    "x":        X_3d[:, 0],
    "y":        X_3d[:, 1],
    "z":        X_3d[:, 2],
    "category": y,
    "title":    labeled_df["title"].values
})

fig = px.scatter_3d(
    plot_df,
    x="x", y="y", z="z",
    color="category",
    hover_data=["title", "category"],
    title="Job Postings — TF-IDF + SVD + UMAP (3D)",
    width=1000, height=700,
    opacity=0.6
)

fig.update_traces(marker=dict(size=3))
fig.update_layout(legend=dict(itemsizing="constant"))
fig.show()

In [0]:
# 2. Check which categories form the large central cluster
#    by coloring only the suspected overlapping categories
central_categories = [
    "Account Management", "Sales", "Business Operations", 
    "Management", "Advertising and Marketing"
]

plot_df["highlight"] = plot_df["category"].apply(
    lambda c: c if c in central_categories else "Other"
)

fig = px.scatter_3d(
    plot_df,
    x="x", y="y", z="z",
    color="highlight",
    hover_data=["title", "category"],
    title="Central Cluster Investigation",
    width=1000, height=700,
    opacity=0.7,
    color_discrete_map={"Other": "#dddddd"}
)
fig.update_traces(marker=dict(size=3))
fig.show()

In [0]:
print("=== k sweep: title boost + distance weighting ===")
X_boosted = (labeled_df["title"] + " " + labeled_df["title"] + " " + labeled_df["text_input"]).values

for k in [3, 5, 7, 11, 15]:
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=10_000, ngram_range=(1,2), 
                                  sublinear_tf=True, min_df=2)),
        ("knn", KNeighborsClassifier(n_neighbors=k, metric="cosine", 
                                     weights="distance"))
    ])
    scores = cross_val_score(pipeline, X_boosted, y, cv=cv, scoring="accuracy")
    print(f"  k={k}:  mean={scores.mean():.3f}  std={scores.std():.3f}")

In [0]:
pipeline_final = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=10_000,
        ngram_range=(1, 2),
        sublinear_tf=True,
        min_df=2
    )),
    ("knn", KNeighborsClassifier(
        n_neighbors=5,
        metric="cosine",
        weights="distance"
    ))
])

# Title-boosted text input
X_final = (labeled_df["title"] + " " + labeled_df["title"] + " " + labeled_df["text_input"]).values
y_final = labeled_df["category"].values

In [0]:
# 1. Full classification report on the winning config
y_pred_final = cross_val_predict(pipeline_final, X_final, y_final, cv=cv)
print(classification_report(y_final, y_pred_final))